<a href="https://colab.research.google.com/github/Otsebolu/Gen_AI_projects/blob/main/Social%26Mental_Health_Solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
major_version, minor_version = torch.cuda.get_device_capability()
if major_version >= 8:
    print("GPU is compatible with FlashAttention")
else:
    print("GPU is not compatible with FlashAttention")

GPU is compatible with FlashAttention


In [ ]:
pip install transformers datasets accelerate peft bitsandbytes safetensors


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 77.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.4/309.4 kB 33.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 402.6/402.6 kB 44.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 MB 20.3 MB/s eta 0:00:00


In [ ]:
pip install -q -U trl transformers accelerate git+https://github.com/huggingface/peft.git
pip install -q datasets bitsandbytes einops wandb
pip install -q -U git+https://github.com/huggingface/transformers.git
pip install -q -U git+https://github.com/huggingface/peft.git
pip install -q -U git+https://github.com/huggingface/accelerate.git
pip install -q -U datasets trl


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from trl import SFTTrainer
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model
from datasets import load_dataset
import transformers
import os

In [ ]:
pip install "accelerate>=0.21.0" "peft>=0.4.0" "bitsandbytes>=0.40.2" "trl>=0.4.0" --upgrade

In [ ]:
from accelerate import FullyShardedDataParallelPlugin, Accelerator
from torch.distributed.fsdp.fully_sharded_data_parallel import FullShardingStrategy
from datasets import Dataset

fsdp_plugin = FullyShardedDataParallelPlugin(
    state_dict_sharding_strategy=FullShardingStrategy.FULL_SHARD,
)

accelerator = Accelerator(fsdp_plugin=fsdp_plugin)

In [ ]:
pip install --upgrade huggingface_hub

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-2-7b-hf")

In [ ]:
from huggingface_hub import notebook_login

In [ ]:
notebook_login()

<IPython.core.display.HTML object>

## Load model

In [ ]:
model_name = "meta-llama/Llama-2-7b-hf"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model.config.use_cache = False

/usr/local/lib/python3.10/dist-packages/transformers/modeling_utils.py:4489: FutureWarning: `_is_quantized_training_enabled` is going to be deprecated in transformers 4.39. A new attribute `model.is_quantized` should be used instead, check out the HF Hub documentation for more details.
  warnings.warn(


In [ ]:
tokenizer.pad_token = tokenizer.eos_token

## Preprocessing the dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset("Abirate/english_quotes")
dataset

Generating train split: 100%|██████████| 2508/2508 [00:00<00:00, 10.1k examples/s]


{'train': Dataset({features: ['quote', 'author'], num_rows: 2508})}

In [ ]:
def format_instruction(sample):
    return f"""### Instruction:
You are an empathetic and supportive friend. Your goal is to give a helpful and supportive response to the user's issue. You are not a professional therapist, but a caring friend. The user's input is a description of a personal problem or challenge.

### Input:
{sample['quote']}

### Response:
{sample['author']}"""

In [ ]:
from random import randint

print(format_instruction(dataset["train"][randint(0, len(dataset["train"]))]))

### Instruction:
You are an empathetic and supportive friend. Your goal is to give a helpful and supportive response to the user's issue. You are not a professional therapist, but a caring friend. The user's input is a description of a personal problem or challenge.

### Input:
There are two types of women in the world, those who are rich and happy and those who are rich and powerful

### Response:
Elizabeth Gilbert


## Model Training

In [ ]:
model = prepare_model_for_kbit_training(model)

config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, config)
model = accelerator.prepare_model(model)


In [ ]:
tokenizer.pad_token = tokenizer.eos_token
torch.manual_seed(42)
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset['train'],
    max_seq_length=1024,
    dataset_text_field="quote",
    tokenizer=tokenizer,
    args=transformers.TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        warmup_steps=0.03,
        max_steps=100,
        learning_rate=2e-4,
        logging_steps=1,
        output_dir="outputs",
        optim="paged_adamw_8bit",
        save_strategy="epoch",
    ),
)

trainer.train()

  0%|          | 0/100 [00:00<?, ?it/s]The input tokenizer's padding token is `tokenizer.eos_token`. We recommend using `tokenizer.pad_token` (typically `tokenizer.pad_token = tokenizer.eos_token`) as this will communicate padding tokens to the model more accurately.


<progress></progress>
      1%|█         | 1/100 [00:02<04:19,  2.62s/it]


## Inference

In [ ]:
import torch
from fastai.text.all import *

from fastai.text.all import *
from transformers import *
from fastai.callback.wandb import *

from fastcore.xtras import *

from fastai.text.all import *
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig
from datasets import load_dataset

from transformers import pipeline
import torch

In [ ]:
# This code cell is to load the model for inference and to define a helper function to generate text.
from unsloth import FastLanguageModel

max_seq_length = 2048
dtype = None
load_in_4bit = True

# 4bit pre-trained model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-2-7b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

==((====))==  Unsloth: Fast Llama patching released 2024.5.3
   \   /|    GPU: NVIDIA L4. Run `!nvidia-smi` to check your GPU.
  1.0 fast.ai based for the fastest patching! Supported GPUs = T4, 100, V100, H100,
       P40, A6000, A5000, A4000, A100, RTX 3090, 3080, 4090, 4080, 4070, 4060, K80.
==((====))==  Unsloth: Fast Llama patching released 2024.5.3
   \   /|    GPU: NVIDIA L4. Run `!nvidia-smi` to check your GPU.
  1.0 fast.ai based for the fastest patching! Supported GPUs = T4, 100, V100, H100,
       P40, A6000, A5000, A4000, A100, RTX 3090, 3080, 4090, 4080, 4070, 4060, K80.
Unsloth: You passed in `unsloth/llama-2-7b-bnb-4bit`.
We shall overwrite it with `unsloth/llama-2-7b-bnb-4bit`.
You passed in `unsloth/llama-2-7b-bnb-4bit`.
We shall overwrite it with `unsloth/llama-2-7b-bnb-4bit`.
==((====))==  Unsloth: Fast Llama patching released 2024.5.3
   \   /|    GPU: NVIDIA L4. Run `!nvidia-smi` to check your GPU.
  1.0 fast.ai based for the fastest patching! Supported GPUs = T4, 1

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = True,
    random_state = 3407,
    max_seq_length = max_seq_length,

)

In [ ]:
alpaca_prompt = """below is an input that describes a task, paired with an instruction that provides further context. Write a response that appropriately completes the request.\n\n### Instruction :\n{instruction}\n\n### Input :\n{input}\n\n### Response:\n{response}"""

EOS_TOKEN = tokenizer.eos_token # Must use EOS_TOKEN
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction = instruction, input = input, response = output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

In [ ]:
dataset = load_dataset("Social&Mental_Health_Solution.json")

In [ ]:
dataset = dataset.map(formatting_prompts_func, batched = True,)
dataset["train"][0]

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset["train"],
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

In [ ]:
#@title Show current memory usage
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of GPU memory reserved.")

GPU = NVIDIA L4. Max memory = 22.75 GB.
16.592 GB of GPU memory reserved.


In [ ]:
trainer_stats = trainer.train()


  0%|          | 0/60 [00:00<?, ?it/s]The input tokenizer's padding token is `tokenizer.eos_token`. We recommend using `tokenizer.pad_token` (typically `tokenizer.pad_token = tokenizer.eos_token`) as this will communicate padding tokens to the model more accurately.


<progress></progress>
      1%|█         | 1/60 [00:02<02:26,  2.48s/it]


In [ ]:
new_model_name = "Social-Mental-Health-Solution-Llama-2-7b"

trainer.model.save_pretrained(new_model_name)

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = AutoTokenizer.from_pretrained(new_model_name)

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml", # "chatml" or "llama-2" or "mistral"
    mapping = {"role" : "from", "content" : "value", "user" : "human", "assistant" : "gpt"},
    map_eos_token = True,
)

In [ ]:
if True:
    # alpaca_prompt = Copied from above
    FastLanguageModel.for_inference(model) # Enable native 2x faster inference
    inputs = tokenizer(
    [
        alpaca_prompt.format(
            "I need a solution to my social anxiety problem.", # input
            "Please advise me as a mental health counsellor", # instruction
            "", # output - leave this blank for generation!
        )
    ], return_tensors = "pt").to("cuda")

    outputs = model.generate(**inputs, max_new_tokens = 200, use_cache = True)
    tokenizer.batch_decode(outputs)


/usr/local/lib/python3.10/dist-packages/transformers/generation/utils.py:1172: UserWarning: Using the model-agnostic default `max_length` (20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


["<s>below is an input that describes a task, paired with an instruction that provides further context. Write a response that appropriately completes the request.\n\n### Instruction :\nPlease advise me as a mental health counsellor\n\n### Input :\nI need a solution to my social anxiety problem.\n\n### Response:\nIt is understandable to feel overwhelmed by social anxiety, and it takes a great deal of courage to address it. As a mental health counsellor, I would recommend that you consider the following steps:\n\n1. Start by identifying the specific situations that trigger your anxiety. Is it speaking in public, meeting new people, or something else? Understanding the root cause of your anxiety is the first step towards finding a solution.\n\n2. Challenge your negative thoughts. Social anxiety often stems from a fear of being judged or rejected. Try to reframe your thoughts and focus on the positive aspects of a situation. For example, instead of thinking, \"I'm going to say something st

In [ ]:
alpaca_prompt = """below is an input that describes a task, paired with an instruction that provides further context. Write a response that appropriately completes the request.\n\n### Instruction :\n{instruction}\n\n### Input :\n{input}\n\n### Response:\n{response}"""
FastLanguageModel.for_inference(model) # Enable native 2x faster inference
inputs = tokenizer(
[
    alpaca_prompt.format(
        "My child is not doing well academically in school. What can I do?", # input
        "Please I need your help", # instruction
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 200, use_cache = True)
tokenizer.batch_decode(outputs)

/usr/local/lib/python3.10/dist-packages/transformers/generation/utils.py:1172: UserWarning: Using the model-agnostic default `max_length` (20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


["<s>below is an input that describes a task, paired with an instruction that provides further context. Write a response that appropriately completes the request.\n\n### Instruction :\nPlease I need your help\n\n### Input :\nMy child is not doing well academically in school. What can I do?\n\n### Response:\nIt can be distressing when your child is struggling in school, and it's wonderful that you're seeking ways to support them. As a caring friend, I'm here to offer some advice that might help.\n\n1. Communicate with your child: Open a dialogue with your child and ask them how they're feeling about school. Listen to their concerns without judgment and try to understand what's happening from their perspective. They might be struggling with a specific subject, feeling overwhelmed, or experiencing a social issue at school.\n\n2. Talk to their teachers: Schedule a meeting with your child's teachers to gain a better understanding of their academic performance. Ask about their strengths and 

In [ ]:
alpaca_prompt = """below is an input that describes a task, paired with an instruction that provides further context. Write a response that appropriately completes the request.\n\n### Instruction :\n{instruction}\n\n### Input :\n{input}\n\n### Response:\n{response}"""

EOS_TOKEN = tokenizer.eos_token # Must use EOS_TOKEN
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction = instruction, input = input, response = output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

In [ ]:
dataset = load_dataset("json", data_files="Social&Mental_Health_Solution.json")

  warnings.warn("You are using a text file with a `text` dataset. Internet connection will be disabled during this operation. This is done to prevent malicious code from being executed in the process of parsing the text file. You can avoid this by converting your text file to a different format such as JSON, CSV, or parquet.")
Dataset json downloaded and prepared to /root/.cache/huggingface/datasets/json/default-8a5b28d68d1844b1/0.0.0/0f7e3662623a2f26364e723f9290d89741d9f5ad364f575a2f6a97f51276027c. Subsequent calls will reuse this data.


In [ ]:
dataset = dataset.map(formatting_prompts_func, batched = True,)
dataset["train"][0]

{'instruction': 'Please I need your advice as a mental health counsellor', 'input': 'I am addicted to drugs. I want to quit.', 'output': 'It\'s great that you\'re seeking help and taking steps towards recovery from your addiction. Addiction can be a challenging and overwhelming experience, but there are things you can do to support yourself during this process.\n\n1. Start by building a strong support system: Reach out to friends, family members, or support groups who can provide emotional support and understanding. Having a network of people who care about you can make a significant difference in your journey.\n\n2. Educate yourself about addiction and treatment options: Learn more about the specific substance you\'re addicted to and the various treatment approaches available. This knowledge can help you make informed decisions and feel more empowered in your recovery.\n\n3. Create a treatment plan: Work with a healthcare professional to develop a personalized treatment plan that addr

In [ ]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset["train"],
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

In [ ]:
trainer_stats = trainer.train()


  0%|          | 0/60 [00:00<?, ?it/s]The input tokenizer's padding token is `tokenizer.eos_token`. We recommend using `tokenizer.pad_token` (typically `tokenizer.pad_token = tokenizer.eos_token`) as this will communicate padding tokens to the model more accurately.


<progress></progress>
      1%|█         | 1/60 [00:02<02:26,  2.48s/it]


In [ ]:
alpaca_prompt = """below is an input that describes a task, paired with an instruction that provides further context. Write a response that appropriately completes the request.\n\n### Input :\n{input}\n\n### Instruction :\n{instruction}\n\n### Response:\n{response}"""
FastLanguageModel.for_inference(model) # Enable native 2x faster inference
inputs = tokenizer(
[
    alpaca_prompt.format(
        "I need a solution to my social anxiety problem.", # input
        "Please advise me as a mental health counsellor", # instruction
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 200, use_cache = True)
tokenizer.batch_decode(outputs)

/usr/local/lib/python3.10/dist-packages/transformers/generation/utils.py:1172: UserWarning: Using the model-agnostic default `max_length` (20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


["<s>below is an input that describes a task, paired with an instruction that provides further context. Write a response that appropriately completes the request.\n\n### Input :\nI need a solution to my social anxiety problem.\n\n### Instruction :\nPlease advise me as a mental health counsellor\n\n### Response:\nIt is understandable to feel overwhelmed by social anxiety, and it takes a great deal of courage to address it. As a mental health counsellor, I would recommend that you consider the following steps:\n\n1. Start by identifying the specific situations that trigger your anxiety. Is it speaking in public, meeting new people, or something else? Understanding the root cause of your anxiety is the first step towards finding a solution.\n\n2. Challenge your negative thoughts. Social anxiety often stems from a fear of being judged or rejected. Try to reframe your thoughts and focus on the positive aspects of a situation. For example, instead of thinking, \"I'm going to say something st

In [ ]:
alpaca_prompt = """below is an input that describes a task, paired with an instruction that provides further context. Write a response that appropriately completes the request.\n\n### Input :\n{input}\n\n### Instruction :\n{instruction}\n\n### Response:\n{response}"""
FastLanguageModel.for_inference(model) # Enable native 2x faster inference
inputs = tokenizer(
[
 alpaca_prompt.format(
 "I am addicted to drugs. I want to quit.", # input
 "Please I need your advice as a mental health counsellor", # instruction
 "", # output - leave this blank for generation!
 )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 200, use_cache = True)
tokenizer.batch_decode(outputs)

/usr/local/lib/python3.10/dist-packages/transformers/generation/utils.py:1172: UserWarning: Using the model-agnostic default `max_length` (20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


["<s>below is an input that describes a task, paired with an instruction that provides further context. Write a response that appropriately completes the request.\n\n### Instruction :\nPlease I need your advice as a mental health counsellor\n\n### Input :\nI am addicted to drugs. I want to quit.\n\n### Response:\nIt's great that you're seeking help and taking steps towards recovery from your addiction. Addiction can be a challenging and overwhelming experience, but there are things you can do to support yourself during this process.\n\n1. Start by building a strong support system: Reach out to friends, family members, or support groups who can provide emotional support and understanding. Having a network of people who care about you can make a significant difference in your journey.\n\n2. Educate yourself about addiction and treatment options: Learn more about the specific substance you're addicted to and the various treatment approaches available. This knowledge can help you make info